# Baseline Random Forest và XGBoost

Chạy thử trên `feature_28d_sample_step4.csv`. Đặt CSV ở `data/processed/` trong repo hoặc sửa `sample_path` bên dưới. Kernel cần có `pandas`, `numpy`, `scikit-learn` và `xgboost`.

Mẫu có 20 lượt đăng ký và chỉ một nhãn dương. Hai mô hình fit trên toàn bộ mẫu để kiểm tra luồng `fit → predict_proba`. Bảng xác suất bên dưới là kết quả trên chính dữ liệu vừa fit.

In [2]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 9.2 MB/s  0:00:06a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 9.2 MB/s  0:00:32 eta 0:00:010:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]


In [ ]:
import sys
from pathlib import Path

# Hỗ trợ mở notebook từ thư mục gốc repo hoặc từ notebooks/.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.models.baseline_data import load_sample
from src.models.train_rf import train_rf
from src.models.train_xgb import train_xgb

#thay sample path bằng đường dẫn tới file .csv
sample_path = repo_root / "data" / "processed" / "feature_28d_sample_step4.csv"
X, y, registration_keys = load_sample(sample_path)
print(f"{len(X)} lượt đăng ký, {X.shape[1]} feature; phân bố nhãn: {y.value_counts().sort_index().to_dict()}")

20 lượt đăng ký, 21 feature; phân bố nhãn: {0: 19, 1: 1}


In [6]:
# OULAD CONTRACT [AI HANDOFF]: chỉ dùng toàn bộ CSV để smoke test.
# Khi có bảng thật, ghép split chung theo id_student rồi chỉ fit trên train.
# Dùng dev để chọn tham số/ngưỡng, giữ test đến lần đánh giá cuối.
rf = train_rf(X, y)
xgb = train_xgb(X, y)

# Giữ khóa ngoài X; chỉ ghép vào bảng kết quả để đối chiếu từng dòng.
# Cả hai mô hình dùng lớp 1 (Withdrawn) ở cột xác suất thứ hai.
smoke_predictions = registration_keys.assign(
    label_withdrawn=y,
    rf_p_withdrawn=rf.predict_proba(X)[:, 1],
    xgb_p_withdrawn=xgb.predict_proba(X)[:, 1],
)
smoke_predictions.head(20)

,id_student,code_module,code_presentation,label_withdrawn,rf_p_withdrawn,xgb_p_withdrawn
0,11391,AAA,2013J,0,0.26,0.053654
1,28400,AAA,2013J,0,0.04,0.053654
2,31604,AAA,2013J,0,0.01,0.053654
3,32885,AAA,2013J,0,0.04,0.053654
4,38053,AAA,2013J,0,0.00,0.053654
5,45462,AAA,2013J,0,0.11,0.053654
6,45642,AAA,2013J,0,0.01,0.053654
7,52130,AAA,2013J,0,0.01,0.053654
8,53025,AAA,2013J,0,0.00,0.053654
9,57506,AAA,2013J,0,0.00,0.053654
